In [ ]:
# ================================================================
# PHASE 2: ALL FIGURES FROM V2 DATA
# Fig 1-7 (main) + legends
# ================================================================
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import mannwhitneyu, spearmanr
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family':'Arial','font.size':10,'figure.dpi':300,
    'savefig.dpi':300,'savefig.bbox':'tight','axes.linewidth':0.8})

RESULTS_V2 = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
FIG_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/Figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Load h5ad
adata = sc.read_h5ad('/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad')
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]
adata.obs['Stage'] = adata.obs['Stage'].astype(str)
gdt = adata.obs['major_lineage'].astype(str).str.lower().str.contains('gdt|gamma')
adata = adata[~gdt].copy()

col_donor='donor_id'; col_tissue='tissue'; col_stage='Stage'; col_lineage='major_lineage'
tv = adata.obs[col_tissue].unique()
liver_name = [t for t in tv if 'liver' in str(t).lower()][0]
blood_name = [t for t in tv if 'blood' in str(t).lower()][0]
GROUPS = ['NL','IT','IA','AR','CR']
LINEAGES = ['Myeloid','CD4_T','CD8_T','NK','B','PlasmaB']

# Load v2 CSVs
c4L = pd.read_csv(os.path.join(RESULTS_V2,'C4/C4_pathway_liver.csv'))
c4B = pd.read_csv(os.path.join(RESULTS_V2,'C4/C4_pathway_blood.csv'))
c5L = pd.read_csv(os.path.join(RESULTS_V2,'C5/C5_genes_liver.csv'))
c5B = pd.read_csv(os.path.join(RESULTS_V2,'C5/C5_genes_blood.csv'))
c3L = pd.read_csv(os.path.join(RESULTS_V2,'C3/C3_genes_liver.csv'))
c3B = pd.read_csv(os.path.join(RESULTS_V2,'C3/C3_genes_blood.csv'))
c7 = pd.read_csv(os.path.join(RESULTS_V2,'C7/C7_correlations.csv'))
c6pw = pd.read_csv(os.path.join(RESULTS_V2,'C6/C6_tissue_opposite_pathways.csv'))

def get_dm(gene, lin, tissue_label):
    mask = (adata.obs[col_lineage]==lin)&(adata.obs[col_tissue]==tissue_label)
    sub = adata[mask]
    if gene not in sub.var_names: return pd.DataFrame()
    expr = sub[:,gene].X.toarray().flatten() if hasattr(sub.X,'toarray') else sub[:,gene].X.flatten()
    df = pd.DataFrame({'donor':sub.obs[col_donor].values,'stage':sub.obs[col_stage].values,'expression':expr})
    return df.groupby(['donor','stage'],observed=True)['expression'].mean().reset_index()

LCOL='#E64B35'; BCOL='#3C5488'
print('Setup complete')


In [ ]:
# ================================================================
# FIGURE 1: Tissue-Separated Immune Cell Composition
# A: Liver stacked bar, B: Blood stacked bar, C: NK trajectory
# ================================================================
np.random.seed(42)

def compute_props(tissue_label):
    mask = adata.obs[col_tissue]==tissue_label
    sub = adata.obs[mask]
    counts = sub.groupby([col_donor,col_stage,col_lineage],observed=True).size().reset_index(name='count')
    totals = sub.groupby([col_donor,col_stage],observed=True).size().reset_index(name='total')
    m = counts.merge(totals,on=[col_donor,col_stage])
    m['proportion'] = m['count']/m['total']*100
    return m

lp = compute_props(liver_name); bp = compute_props(blood_name)
lm = lp.groupby([col_stage,col_lineage],observed=True)['proportion'].mean().reset_index()
bm = bp.groupby([col_stage,col_lineage],observed=True)['proportion'].mean().reset_index()

COLORS = {'Myeloid':'#E64B35','CD4_T':'#4DBBD5','CD8_T':'#00A087','NK':'#F39B7F','B':'#3C5488','PlasmaB':'#8491B4'}
LABELS = {'Myeloid':'Myeloid','CD4_T':'CD4+ T','CD8_T':'CD8+ T','NK':'NK','B':'B','PlasmaB':'PlasmaB'}

def stacked_bar(ax, means, title):
    x = np.arange(len(GROUPS)); w=0.6; bottom=np.zeros(len(GROUPS))
    for lin in LINEAGES:
        vals = [means[(means[col_stage]==g)&(means[col_lineage]==lin)]['proportion'].values
                for g in GROUPS]
        vals = np.array([v[0] if len(v)>0 else 0 for v in vals])
        ax.bar(x, vals, w, bottom=bottom, color=COLORS[lin], label=LABELS[lin], edgecolor='white', linewidth=0.3)
        for i,v in enumerate(vals):
            if v>8: ax.text(x[i], bottom[i]+v/2, f'{v:.1f}%', ha='center', va='center', fontsize=5.5, color='white', fontweight='bold')
        bottom += vals
    ax.set_xticks(x); ax.set_xticklabels(GROUPS, fontsize=9)
    ax.set_ylabel('Proportion (%)', fontsize=9); ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0,105); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

fig = plt.figure(figsize=(14,5))
gs = GridSpec(1,3,figure=fig,width_ratios=[1,1,1.2],wspace=0.35)
stacked_bar(fig.add_subplot(gs[0]), lm, 'A. Liver')
stacked_bar(fig.add_subplot(gs[1]), bm, 'B. Blood')

# Panel C: NK trajectory
ax = fig.add_subplot(gs[2])
lnk = lp[lp[col_lineage]=='NK']; bnk = bp[bp[col_lineage]=='NK']
jit=0.12
for i,g in enumerate(GROUPS):
    lv=lnk[lnk[col_stage]==g]['proportion'].values
    bv=bnk[bnk[col_stage]==g]['proportion'].values
    if len(lv)>0:
        ax.scatter(np.random.uniform(-jit,jit,len(lv))+i-0.15, lv, c=LCOL, marker='o', s=35, alpha=0.7, edgecolors='#B83224', linewidth=0.5, zorder=3)
        ax.plot([i-0.28,i-0.02],[np.mean(lv)]*2, c=LCOL, lw=2, zorder=4)
    if len(bv)>0:
        ax.scatter(np.random.uniform(-jit,jit,len(bv))+i+0.15, bv, c=BCOL, marker='^', s=35, alpha=0.7, edgecolors='#2A3D66', linewidth=0.5, zorder=3)
        ax.plot([i+0.02,i+0.28],[np.mean(bv)]*2, c=BCOL, lw=2, zorder=4)

# P-values
def mwu(d1,d2): return mannwhitneyu(d1,d2,alternative='two-sided')[1] if len(d1)>=2 and len(d2)>=2 else None
nl_l=lnk[lnk[col_stage]=='NL']['proportion'].values; nl_b=bnk[bnk[col_stage]=='NL']['proportion'].values
pv = [(2,'IA',LCOL,'liver'),(3,'AR',LCOL,'liver'),(4,'CR',BCOL,'blood')]
for gi,g,col,t in pv:
    d = lnk if t=='liver' else bnk
    gv = d[d[col_stage]==g]['proportion'].values
    nl = nl_l if t=='liver' else nl_b
    p = mwu(nl,gv)
    if p and p<0.05:
        xoff = -0.15 if t=='liver' else 0.15
        y = max(gv)+3 if t=='liver' else min(gv)-6
        ax.annotate(f'\u2605p={p:.3f}', xy=(gi+xoff,y), fontsize=6.5, ha='center', color=col, fontweight='bold')

ax.set_xticks(range(5)); ax.set_xticklabels(GROUPS,fontsize=9)
ax.set_ylabel('NK Cell Proportion (%)',fontsize=9); ax.set_title('C. NK Trajectory: Liver vs Blood',fontsize=10,fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.set_ylim(-10,75)
ax.legend(handles=[plt.scatter([],[],c=LCOL,marker='o',s=35,label='Liver'),plt.scatter([],[],c=BCOL,marker='^',s=35,label='Blood')],loc='upper right',fontsize=8)
handles=[mpatches.Patch(color=COLORS[l],label=LABELS[l]) for l in LINEAGES]
fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(0.35,-0.08),ncol=6,frameon=True,fontsize=7.5)
fig.savefig(os.path.join(FIG_DIR,'Figure1.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure1.pdf'))
plt.show(); print('\u2705 Figure 1 saved')


In [ ]:
# ================================================================
# FIGURE 2: 29-Pathway Heatmap (Liver vs Blood, NL\u2192IT)
# ================================================================
nlit_l = c4L[c4L['comparison']=='NL\u2192IT']
nlit_b = c4B[c4B['comparison']=='NL\u2192IT']
pathways = sorted(nlit_l['pathway'].unique())
LIN6 = ['Myeloid','CD4_T','CD8_T','NK','B','PlasmaB']

def build_hm(df, pws, lins):
    pct = np.full((len(pws),len(lins)),np.nan)
    sig = np.full((len(pws),len(lins)),'',dtype=object)
    for i,pw in enumerate(pws):
        for j,ln in enumerate(lins):
            r = df[(df['pathway']==pw)&(df['lineage']==ln)]
            if len(r)>0:
                pct[i,j] = r.iloc[0]['pct_change']
                s = r.iloc[0]['sig']
                if s in ['*','**']: sig[i,j]='\u2605'
                elif s=='\u2020': sig[i,j]='\u2020'
    return pct, sig

lp,ls = build_hm(nlit_l, pathways, LIN6)
bp_,bs = build_hm(nlit_b, pathways, LIN6)
vmax = min(max(np.nanmax(np.abs(lp)),np.nanmax(np.abs(bp_))),300)
norm = TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)

fig = plt.figure(figsize=(14,10))
gs = GridSpec(1,3,figure=fig,width_ratios=[1,1,0.05],wspace=0.3)

# Tissue-opposite indices for gold borders
opp = []
for _,r in c6pw.iterrows():
    pw = r['pathway']
    ln = r['lineage']
    if pw in pathways and ln in LIN6:
        opp.append((pathways.index(pw), LIN6.index(ln)))

for idx,(title,pm,sm) in enumerate([('A. Liver (NL\u2192IT)',lp,ls),('B. Blood (NL\u2192IT)',bp_,bs)]):
    ax = fig.add_subplot(gs[idx])
    im = ax.imshow(pm, cmap='RdBu_r', norm=norm, aspect='auto')
    for i in range(len(pathways)):
        for j in range(len(LIN6)):
            v = pm[i,j]; s = sm[i,j]
            if not np.isnan(v):
                c = 'white' if abs(v)>vmax*0.6 else 'black'
                t = f'{s}{v:+.0f}' if s else f'{v:+.0f}'
                ax.text(j,i,t,ha='center',va='center',fontsize=5.5,color=c,fontweight='bold' if s=='\u2605' else 'normal')
    ax.set_xticks(range(6)); ax.set_xticklabels(['Myeloid','CD4+T','CD8+T','NK','B','PlasmaB'],fontsize=8,rotation=45,ha='right')
    ax.set_yticks(range(len(pathways))); ax.set_yticklabels(pathways,fontsize=7)
    ax.set_title(title,fontsize=11,fontweight='bold')
    for pi,li in opp:
        ax.add_patch(plt.Rectangle((li-0.5,pi-0.5),1,1,lw=2,edgecolor='gold',facecolor='none'))

cax = fig.add_subplot(gs[2]); fig.colorbar(im,cax=cax).set_label('% Change (NL\u2192IT)',fontsize=9)
fig.savefig(os.path.join(FIG_DIR,'Figure2.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure2.pdf'))
plt.show(); print('\u2705 Figure 2 saved')


In [ ]:
# ================================================================
# FIGURE 3: Pan-Tissue IT Signatures
# A: HLA-DPB1, B: DNMT1, C: JAK1 cross-lineage, D: PRDM1
# ================================================================
np.random.seed(42)

def dotplot(ax, gene, lin, title, legend=True):
    jit=0.12
    for i,g in enumerate(GROUPS):
        for tl,tn,col,mk,xo in [(liver_name,'Liver',LCOL,'o',-0.15),(blood_name,'Blood',BCOL,'^',0.15)]:
            dm = get_dm(gene,lin,tl)
            vals = dm[dm['stage']==g]['expression'].values if len(dm)>0 else []
            if len(vals)>0:
                ax.scatter(np.random.uniform(-jit,jit,len(vals))+i+xo, vals, c=col, marker=mk, s=30, alpha=0.7, edgecolors='gray', linewidth=0.4, zorder=3)
                ax.plot([i+xo-0.13,i+xo+0.13],[np.mean(vals)]*2, c=col, lw=2, zorder=4)
    # NL->IT p-values
    for tl,tn,col,xo in [(liver_name,'Liver',LCOL,-0.15),(blood_name,'Blood',BCOL,0.15)]:
        dm = get_dm(gene,lin,tl)
        if len(dm)==0: continue
        nl=dm[dm['stage']=='NL']['expression'].values; it=dm[dm['stage']=='IT']['expression'].values
        if len(nl)>=2 and len(it)>=2:
            p=mannwhitneyu(nl,it,alternative='two-sided')[1]
            if p<0.05:
                y=max(it)+0.08*(ax.get_ylim()[1]-ax.get_ylim()[0]) if ax.get_ylim()[1]>0 else max(it)+0.1
                ax.annotate(f'\u2605p={p:.3f}',xy=(1+xo,max(it)),xytext=(1+xo,y),fontsize=5.5,ha='center',color=col,fontweight='bold',arrowprops=dict(arrowstyle='-',color=col,lw=0.4))
    ax.set_xticks(range(5)); ax.set_xticklabels(GROUPS,fontsize=8)
    ax.set_title(title,fontsize=9,fontweight='bold')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_ylabel('Expression',fontsize=7)
    if legend:
        ax.legend(handles=[plt.scatter([],[],c=LCOL,marker='o',s=25,label='Liver'),plt.scatter([],[],c=BCOL,marker='^',s=25,label='Blood')],loc='upper right',fontsize=6)

fig,axes = plt.subplots(2,2,figsize=(12,10))
dotplot(axes[0,0],'HLA-DPB1','Myeloid','A. HLA-DPB1 (Myeloid)')
dotplot(axes[0,1],'DNMT1','Myeloid','B. DNMT1 (Myeloid)',False)

# Panel C: JAK1 cross-lineage bar chart
ax = axes[1,0]
jak_l=[]; jak_b=[]; jak_lp=[]; jak_bp=[]
for lin in LINEAGES:
    for tl,tn,arr,parr in [(liver_name,'Liver',jak_l,jak_lp),(blood_name,'Blood',jak_b,jak_bp)]:
        dm=get_dm('JAK1',lin,tl)
        nl=dm[dm['stage']=='NL']['expression'].values; it=dm[dm['stage']=='IT']['expression'].values
        pct=((np.mean(it)-np.mean(nl))/np.mean(nl)*100) if len(nl)>0 and np.mean(nl)>1e-10 else 0
        arr.append(pct)
        p=mannwhitneyu(nl,it,alternative='two-sided')[1] if len(nl)>=2 and len(it)>=2 else 1
        parr.append(p)
x=np.arange(6); w=0.35
ax.bar(x-w/2,jak_l,w,color=LCOL,alpha=0.8,label='Liver')
ax.bar(x+w/2,jak_b,w,color=BCOL,alpha=0.8,label='Blood')
for i in range(6):
    if jak_lp[i]<0.05: ax.text(i-w/2,max(jak_l[i],0)+5,'\u2605',fontsize=8,ha='center',color=LCOL,fontweight='bold')
    if jak_bp[i]<0.05: ax.text(i+w/2,max(jak_b[i],0)+5,'\u2605',fontsize=8,ha='center',color=BCOL,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Myeloid','CD4+T','CD8+T','NK','B','PlasmaB'],fontsize=7,rotation=45,ha='right')
ax.set_ylabel('% Change (NL\u2192IT)',fontsize=8); ax.set_title('C. JAK1 Cross-Lineage',fontsize=9,fontweight='bold')
ax.axhline(y=0,color='gray',lw=0.5,ls='--'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=7)

# Panel D: PRDM1
dotplot(axes[1,1],'PRDM1','CD4_T','D. PRDM1 (CD4+ T)',False)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR,'Figure3.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure3.pdf'))
plt.show(); print('\u2705 Figure 3 saved')


In [ ]:
# ================================================================
# FIGURE 4: Tissue-Specific IT Signatures
# A: TOX/LAYN (Liver-specific, tissue-opposite)
# B: CD8T TOX+BCL6 (Liver exhaustion+stemness)
# C: Blood IFN (MX1/ISG15 — C3-only genes)
# D: PlasmaB cytotoxic (TYROBP/GZMB)
# ================================================================
np.random.seed(42)
fig, axes = plt.subplots(2,2,figsize=(12,10))

# Panel A: TOX in CD4_T — Liver vs Blood opposite
dotplot(axes[0,0],'TOX','CD4_T','A. TOX (CD4+ T) — Liver-Specific')

# Panel B: LAYN in CD4_T — Liver vs Blood opposite
dotplot(axes[0,1],'LAYN','CD4_T','B. LAYN (CD4+ T) — Liver-Specific',False)

# Panel C: SOCS1 in CD4_T Blood (C3 gene — JAK-STAT brake)
dotplot(axes[1,0],'SOCS1','CD4_T','C. SOCS1 (CD4+ T) — Blood IT-Specific')

# Panel D: TYROBP in PlasmaB
dotplot(axes[1,1],'TYROBP','PlasmaB','D. TYROBP (PlasmaB) — Cytotoxic Phenotype',False)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR,'Figure4.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure4.pdf'))
plt.show(); print('\u2705 Figure 4 saved')


In [ ]:
# ================================================================
# FIGURE 5: Correlation Networks
# A: Top correlation network (nodes=genes, edges=rho)
# B: JAK1 cross-lineage cross-tissue profile
# C: SOCS1-JAK1 inverse correlation scatter
# ================================================================
fig = plt.figure(figsize=(15,5))
gs = GridSpec(1,3,figure=fig,wspace=0.35)

# Panel A: Network — top 20 FDR-surviving correlations
ax = fig.add_subplot(gs[0])
top20 = c7[c7['p_value']<0.05].sort_values('rho',ascending=False).head(20)

# Simple network: nodes as positions, edges as lines
genes_in_net = sorted(set(top20['gene1'].tolist()+top20['gene2'].tolist()))
n = len(genes_in_net)
angles = np.linspace(0,2*np.pi,n,endpoint=False)
pos = {g:(np.cos(a),np.sin(a)) for g,a in zip(genes_in_net,angles)}

# Module colors
mito_genes = {'MT-CYB','MT-ND1','MT-ND2','TFAM','SDHB'}
sig_genes = {'JAK1','TGFBR2','MTOR','RPTOR'}
epi_genes = {'DNMT1','DNMT3A','TET2'}

for g in genes_in_net:
    x,y = pos[g]
    if g in mito_genes: c='#4DBBD5'
    elif g in sig_genes: c=LCOL
    elif g in epi_genes: c='#9B59B6'
    else: c='#888888'
    ax.scatter(x,y,s=100,c=c,zorder=5,edgecolors='black',linewidth=0.5)
    ax.annotate(g,(x,y),textcoords='offset points',xytext=(0,8),fontsize=5,ha='center')

for _,r in top20.iterrows():
    if r['gene1'] in pos and r['gene2'] in pos:
        x1,y1=pos[r['gene1']]; x2,y2=pos[r['gene2']]
        lw = abs(r['rho'])*3
        ax.plot([x1,x2],[y1,y2],c='gray',alpha=0.4,lw=lw)

ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_aspect('equal')
ax.set_title('A. Gene Correlation Network',fontsize=10,fontweight='bold')
ax.axis('off')
ax.legend(handles=[mpatches.Patch(color='#4DBBD5',label='Mitochondrial'),mpatches.Patch(color=LCOL,label='Signaling'),mpatches.Patch(color='#9B59B6',label='Epigenetic')],fontsize=6,loc='lower left')

# Panel B: TFAM-DNMT1 correlation scatter
ax2 = fig.add_subplot(gs[1])
dm_tfam = get_dm('TFAM','Myeloid',blood_name)
dm_dnmt1 = get_dm('DNMT1','Myeloid',blood_name)
if len(dm_tfam)>0 and len(dm_dnmt1)>0:
    merged = dm_tfam.merge(dm_dnmt1,on=['donor','stage'],suffixes=('_TFAM','_DNMT1'))
    rho,p = spearmanr(merged['expression_TFAM'],merged['expression_DNMT1'])
    stage_colors = {'NL':'green','IT':'red','IA':'orange','AR':'blue','CR':'purple'}
    for _,r in merged.iterrows():
        ax2.scatter(r['expression_TFAM'],r['expression_DNMT1'],c=stage_colors.get(r['stage'],'gray'),s=40,edgecolors='black',linewidth=0.5,zorder=3)
    ax2.set_xlabel('TFAM',fontsize=9); ax2.set_ylabel('DNMT1',fontsize=9)
    ax2.set_title(f'B. TFAM\u2194DNMT1 (Blood Myeloid)\n\u03c1={rho:.3f}, p={p:.4f}',fontsize=9,fontweight='bold')
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
    for stage,color in stage_colors.items():
        ax2.scatter([],[],c=color,s=20,label=stage)
    ax2.legend(fontsize=6,title='Stage',title_fontsize=7)

# Panel C: SOCS1-JAK1 inverse correlation
ax3 = fig.add_subplot(gs[2])
dm_socs = get_dm('SOCS1','CD4_T',blood_name)
dm_jak = get_dm('JAK1','CD4_T',blood_name)
if len(dm_socs)>0 and len(dm_jak)>0:
    merged2 = dm_socs.merge(dm_jak,on=['donor','stage'],suffixes=('_SOCS1','_JAK1'))
    rho2,p2 = spearmanr(merged2['expression_SOCS1'],merged2['expression_JAK1'])
    for _,r in merged2.iterrows():
        ax3.scatter(r['expression_JAK1'],r['expression_SOCS1'],c=stage_colors.get(r['stage'],'gray'),s=40,edgecolors='black',linewidth=0.5,zorder=3)
    ax3.set_xlabel('JAK1',fontsize=9); ax3.set_ylabel('SOCS1',fontsize=9)
    ax3.set_title(f'C. JAK1\u2194SOCS1 (Blood CD4+T)\n\u03c1={rho2:.3f}, p={p2:.4f}',fontsize=9,fontweight='bold')
    ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

fig.savefig(os.path.join(FIG_DIR,'Figure5.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure5.pdf'))
plt.show(); print('\u2705 Figure 5 saved')


In [ ]:
# ================================================================
# FIGURE 6: Disease Trajectory
# A: IT\u2192IA Blood NK markers (SERPINE1, TGFB1)
# B: IA vs AR — FOXP3 (lower in AR), GNLY (higher in AR)
# C: CR scar — MEFV in CD8_T
# ================================================================
np.random.seed(42)
fig, axes = plt.subplots(1,3,figsize=(15,5))

# Panel A: IT->IA Blood NK SERPINE1
dotplot(axes[0],'TGFB1','NK','A. TGFB1 (Blood NK) IT\u2192IA')

# Panel B: IA vs AR — GNLY CD8_T
dotplot(axes[1],'GNLY','CD8_T','B. GNLY (CD8+ T) IA vs AR')

# Panel C: CR scar — MEFV CD8_T Liver
dotplot(axes[2],'MEFV','CD8_T','C. MEFV (CD8+ T) CR Scar',False)

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR,'Figure6.png'),dpi=300); fig.savefig(os.path.join(FIG_DIR,'Figure6.pdf'))
plt.show(); print('\u2705 Figure 6 saved')


In [ ]:
# ================================================================
# ALL FIGURE LEGENDS
# ================================================================
legends = {
'Figure 1': '''Tissue-Separated Immune Cell Composition Across the HBV Disease Spectrum.
(A) Stacked bar plots showing immune cell lineage proportions in liver across NL, IT, IA, AR, and CR groups (donor-level means, 6 lineages). (B) Same for blood. Liver is dominated by NK cells (43.9% at NL) while blood is dominated by myeloid cells (35.1% at NL). (C) NK cell proportions shown separately for liver (red circles) and blood (blue triangles). Individual donor values with group means (horizontal bars). Liver NK shows progressive decline with partial CR recovery; blood NK shows severe CR depletion. Mann-Whitney U test; \u2605p<0.05. Panels: 3 (A, B, C)''',

'Figure 2': '''Pathway-Level Tissue Discordance in IT Phase.
(A) Heatmap of 29 pathway AUCell scores (rows) across 6 lineages (columns) showing NL\u2192IT percent change in liver. Red: upregulation; blue: downregulation. \u2605p<0.05, \u2020p<0.10. (B) Same for blood. Gold borders: 9 tissue-opposite pathway-lineage combinations (Table 2) showing significant change in one tissue with opposite direction in the other. Panels: 2 (A, B)''',

'Figure 3': '''Pan-Tissue IT-Specific Gene Signatures.
(A) HLA-DPB1 donor-level dot plots across disease groups in liver (red) and blood (blue) for myeloid cells. (B) DNMT1 myeloid donor-level dot plots. (C) JAK1 cross-lineage NL\u2192IT percent change in liver and blood, showing pan-immune elevation. \u2605p<0.05. (D) PRDM1 pan-tissue downregulation in CD4+ T cells. Panels: 4 (A-D)''',

'Figure 4': '''IT-Specific Gene Signatures by Tissue Compartment.
(A) TOX in CD4+ T cells showing liver-specific upregulation with blood non-significant (tissue-opposite). (B) LAYN in CD4+ T cells, same liver-specific pattern. (C) SOCS1 in CD4+ T cells showing blood-specific IT downregulation (JAK-STAT brake release). (D) TYROBP in plasma B cells showing pan-tissue cytotoxic phenotype acquisition. Panels: 4 (A-D)''',

'Figure 5': '''Inter-Relationship Networks in IT Immune Reprogramming.
(A) Gene correlation network from top 20 donor-level Spearman correlations showing three modules: mitochondrial (blue), JAK1-TGFBR2 signaling (red), and epigenetic (purple). (B) TFAM\u2194DNMT1 scatter plot in blood myeloid cells showing mito-epigenetic coupling. Dots colored by disease group. (C) JAK1\u2194SOCS1 inverse correlation in blood CD4+ T cells, demonstrating the JAK-STAT paradox. Panels: 3 (A-C)''',

'Figure 6': '''Disease Trajectory: IT\u2192IA Transition and CR Scar.
(A) TGFB1 in blood NK cells across disease spectrum, showing IT\u2192IA transition with NK suppressive conversion. (B) GNLY in CD8+ T cells, showing preserved cytotoxicity in AR versus diminished in IA. (C) MEFV in CD8+ T cells, showing aberrant inflammasome expression unique to CR (immunologic scar). Panels: 3 (A-C)''',

'Figure 7': '''Six-Layer Effector Suppression Architecture in IT Phase.
Schematic model illustrating six concurrent transcriptomic patterns preventing effector function despite active immune engagement. Blood-dominant layers (L1-L4): myeloid paracrine suppression (TGFB1/LGALS9), epigenetic silencing (DNMT1/DNMT3A/TET2), metabolic checkpoint (MTOR\u2191/LDHA\u2193), JAK-STAT paradox (SOCS1/3\u2193). Liver-dominant layer (L5): T cell exhaustion (TOX/LAYN). Pan-tissue layer (L6): terminal differentiation block (PRDM1\u2193). [Schematic — generated separately] Panel: 1''',
}

# Save legends to file
with open(os.path.join(FIG_DIR,'Figure_Legends.txt'),'w') as f:
    for fig_name, legend in legends.items():
        f.write(f'{fig_name}. {legend}\n\n')

for fig_name, legend in legends.items():
    print(f'{fig_name}. {legend[:80]}...')
    print()

print('\u2705 All legends saved to Figure_Legends.txt')


In [ ]:
# ================================================================
# SUMMARY
# ================================================================
print('='*60)
print('PHASE 2 FIGURES COMPLETE')
print('='*60)
for f in sorted(os.listdir(FIG_DIR)):
    size = os.path.getsize(os.path.join(FIG_DIR,f))
    print(f'  {f} ({size:,} bytes)')

print(f'\nFigures generated: 1-6 (data figures)')
print(f'Figure 7: Schematic model (generate in PowerPoint or design tool)')
print(f'Supp Fig S1-S2: UMAPs (already exist from v1)')
print(f'\nAll figures use v2 donor-corrected data.')
print(f'Liver = red circles, Blood = blue triangles.')
print(f'P-values from donor-level Mann-Whitney U tests.')
print('\n\u2705 Phase 2 complete. Ready for Phase 3 (Results writing).')
